<a href="https://colab.research.google.com/github/siddhartha-sai-17/Gen-AI/blob/main/Smart_Medical_Question_Answering_system_using_LORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 1: Install Required Libraries

**Objective**

Prepare the environment for LoRA-based fine-tuning by installing the necessary libraries.

In [1]:
pip install Transformers Datasets PEFT Accelerate BitsAndBytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.9 MB/s eta 0:00:00


In [2]:
# Upgrade torchao to a compatible version to resolve the ImportError during PEFT model creation
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## Task 2: Load a Pretrained LLM

**Objective**

Load a pretrained TinyLlama model.

**Explanation:**

### Why use a pretrained model?

Using a pretrained Large Language Model (LLM) offers several significant advantages over training a model from scratch, especially in scenarios like building a medical question-answering system:

1.  **Reduced Computational Cost:** Training an LLM from scratch is incredibly resource-intensive, requiring massive datasets, vast computational power (hundreds or thousands of GPUs for weeks or months), and substantial energy consumption. Pretrained models allow us to leverage this extensive initial training without incurring the full cost.
2.  **Faster Development:** Starting with a pretrained model significantly shortens the development cycle. Instead of building a model from the ground up, you can immediately begin fine-tuning it for your specific task.
3.  **Better Performance (Transfer Learning):** Pretrained LLMs have learned rich, general-purpose linguistic patterns, factual knowledge, and reasoning abilities from diverse and enormous text corpora. This foundational knowledge, acquired during pretraining, is highly transferable to new, related tasks. Even with limited task-specific data, a fine-tuned pretrained model often outperforms a model trained from scratch on that same limited data.
4.  **Handles Data Scarcity:** For specialized domains like medicine, obtaining large, high-quality, labeled datasets can be challenging and expensive. Pretrained models can perform well even with smaller domain-specific datasets through fine-tuning, as they already possess a broad understanding of language.
5.  **Access to State-of-the-Art Architectures:** Pretrained models are often based on the latest and most effective neural network architectures (like Transformers), which would be complex and time-consuming to implement and optimize from scratch.

### Difference between training from scratch and fine-tuning:

*   **Training from scratch:**
    *   Involves initializing a neural network with random weights and training it on a large dataset from the very beginning.
    *   Requires vast amounts of data, computational resources, and time to converge to a good performance.
    *   The model learns all features, patterns, and representations from the ground up.
    *   Typically reserved for novel tasks or when no suitable pretrained models exist.

*   **Fine-tuning (using LoRA):**
    *   Starts with a pre-trained model (like TinyLlama) whose weights have already been optimized on a massive general-purpose dataset.
    *   For **traditional fine-tuning**, all or most of the pretrained model's layers are further trained on a smaller, task-specific dataset. This adjusts the model's learned representations to the nuances of the new task.
    *   For **LoRA (Low-Rank Adaptation) fine-tuning**, instead of updating all the original weights of the pretrained model, new, smaller, low-rank matrices are injected into the transformer layers. Only these small matrices are trained, while the original pretrained model weights remain frozen. This dramatically reduces the number of trainable parameters, leading to:
        *   **Much faster training:** Fewer parameters to update.
        *   **Less memory usage:** Especially beneficial for large models on consumer-grade GPUs.
        *   **Smaller storage footprint:** The fine-tuned adapter weights are tiny compared to the full model, making it easy to store and switch between multiple fine-tuned versions of a base model.
        *   **Reduced catastrophic forgetting:** By keeping the original weights frozen, the model is less likely to forget its general knowledge learned during pretraining.

In summary, fine-tuning with LoRA allows us to efficiently adapt a powerful, generally intelligent LLM to a specific domain (medical QA) with minimal resources and excellent results, bypassing the immense challenges of training from scratch.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Define the model name
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # Use bfloat16 for memory efficiency and speed
    device_map="auto"             # Automatically maps the model to available devices (e.g., GPU)
)

print(f"Model '{model_name}' loaded successfully.")
print(f"Number of parameters: {model.num_parameters() / 1e9:.2f} billion")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' loaded successfully.
Number of parameters: 1.10 billion


In [4]:
# Set the pad_token_id for the tokenizer
# Some tokenizers do not have a default pad_token, which can cause issues during batching.
# Setting it to eos_token_id is a common practice for causal language models.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer pad_token set to: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

Tokenizer pad_token set to: '</s>' (ID: 2)


## Task 3: Load and Prepare Dataset

**Objective**

Load a suitable medical question-answering dataset and prepare it for fine-tuning.

**Dataset Choice Explanation:**

Due to persistent issues with directly accessing medical question-answering datasets from the Hugging Face Hub, we will create a **small synthetic medical question-answering dataset** for demonstration purposes. This allows us to proceed with the LoRA fine-tuning process without being blocked by external data availability.

This synthetic dataset will mimic a simple question-answer format relevant to medical contexts.

**Preparation Steps:**

1.  **Create the synthetic dataset:** Generate a list of sample medical questions and answers.
2.  **Tokenize the data:** Convert the text questions and answers into numerical tokens that the LLM can understand.
3.  **Format for LLM:** Structure the tokenized inputs into a format suitable for causal language models, typically in a conversational turn structure (e.g., `User: <question> Assistant: <answer>`).
4.  **Padding and Truncation:** Ensure all input sequences have a uniform length, either by padding shorter sequences or truncating longer ones, to enable batch processing.

In [5]:
from datasets import Dataset

# Create a small synthetic medical question-answering dataset
data = {
    "question": [
        "What is the function of red blood cells?",
        "What are the common symptoms of the flu?",
        "How is diabetes managed?",
        "What is hypertension?",
        "What is the purpose of a vaccine?",
        "What is penicillin used for?",
        "What causes migraines?",
        "How does the heart pump blood?",
        "What is CPR?",
        "What are antibiotics?"
    ],
    "answer": [
        "Red blood cells carry oxygen from the lungs to all parts of the body.",
        "Common symptoms of the flu include fever, cough, sore throat, muscle aches, and fatigue.",
        "Diabetes is managed through diet, exercise, medication (like insulin), and regular monitoring of blood sugar levels.",
        "Hypertension, or high blood pressure, is a condition in which the force of the blood against the artery walls is too high.",
        "A vaccine stimulates the immune system to produce antibodies against a specific disease, providing immunity.",
        "Penicillin is an antibiotic used to treat various bacterial infections.",
        "Migraines are often caused by a combination of genetic and environmental factors, with specific triggers varying among individuals.",
        "The heart pumps blood by contracting its muscular walls, pushing blood through the circulatory system.",
        "CPR (Cardiopulmonary Resuscitation) is an emergency procedure that combines chest compressions and artificial ventilation to manually preserve brain function until further measures are taken.",
        "Antibiotics are medications that destroy or slow down the growth of bacteria."
    ]
}

dataset = Dataset.from_dict(data)

print(f"Created a synthetic dataset with {len(dataset)} examples.")
print("Example entry:")
print(dataset[0])

Created a synthetic dataset with 10 examples.
Example entry:
{'question': 'What is the function of red blood cells?', 'answer': 'Red blood cells carry oxygen from the lungs to all parts of the body.'}


In [6]:
def formatting_function(example):
    # Format the input and target for the LLM
    # Using a chat format similar to how TinyLlama was fine-tuned
    text = f"<|im_start|>user\n{example['question']}<|im_end|>\n<|im_start|>assistant\n{example['answer']}<|im_end|>"

    # Tokenize the formatted text
    # The tokenizer will handle padding and truncation later in the data collator
    tokenized_output = tokenizer(text, truncation=True, max_length=512)

    # Add 'labels' which are the same as 'input_ids' for causal language modeling
    # The loss function will mask out the input tokens during training
    tokenized_output["labels"] = tokenized_output["input_ids"]
    return tokenized_output

# Apply the formatting and tokenization function to the dataset
tokenized_dataset = dataset.map(formatting_function, batched=False)

print("Tokenized dataset example:")
print(tokenized_dataset[0])
print(f"Decoded example: {tokenizer.decode(tokenized_dataset[0]['input_ids'])}")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenized dataset example:
{'question': 'What is the function of red blood cells?', 'answer': 'Red blood cells carry oxygen from the lungs to all parts of the body.', 'input_ids': [1, 529, 29989, 326, 29918, 2962, 29989, 29958, 1792, 13, 5618, 338, 278, 740, 310, 2654, 10416, 9101, 29973, 29966, 29989, 326, 29918, 355, 29989, 29958, 13, 29966, 29989, 326, 29918, 2962, 29989, 29958, 465, 22137, 13, 9039, 10416, 9101, 8677, 288, 28596, 515, 278, 301, 3085, 304, 599, 5633, 310, 278, 3573, 19423, 29989, 326, 29918, 355, 29989, 29958], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1, 529, 29989, 326, 29918, 2962, 29989, 29958, 1792, 13, 5618, 338, 278, 740, 310, 2654, 10416, 9101, 29973, 29966, 29989, 326, 29918, 355, 29989, 29958, 13, 29966, 29989, 326, 29918, 2962, 29989, 29958, 465, 22137, 13, 9039, 10416, 9101, 8677, 288, 28

## Task 4: Configure LoRA and Training Arguments

**Objective**

Set up the Low-Rank Adaptation (LoRA) configuration and define the training arguments for the fine-tuning process.

**Explanation:**

### LoRA Configuration (PEFT `LoraConfig`):

LoRA works by injecting small, trainable matrices into the transformer layers of the pretrained model while keeping the original model weights frozen. `LoraConfig` from the PEFT (Parameter-Efficient Fine-Tuning) library allows us to specify *which* parts of the model should be adapted and with what parameters.

Key parameters for `LoraConfig` often include:

*   `r`: The rank of the update matrices. A smaller `r` means fewer trainable parameters, faster training, and less memory usage, but potentially less expressive power. A common starting point is 8, 16, or 32.
*   `lora_alpha`: LoRA scaling factor. A higher `lora_alpha` gives more weight to the LoRA updates. Typically, it's set to `r` or a multiple of `r`.
*   `lora_dropout`: The dropout probability for the LoRA layers to prevent overfitting.
*   `target_modules`: A list of regular expressions or module names indicating which layers of the base model should be targeted for LoRA. For causal language models like TinyLlama, targeting attention projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and sometimes feed-forward layers is common.
*   `bias`: Specifies if bias parameters should be trained. Often set to 'none' for LoRA.
*   `task_type`: Specifies the type of task, e.g., `CAUSAL_LM` for generative models.

### Training Arguments (`TrainingArguments`):

`TrainingArguments` from the Hugging Face `transformers` library defines all the hyperparameters for the training process. These include:

*   `output_dir`: Directory to save checkpoints and logs.
*   `num_train_epochs`: Number of full passes through the training data.
*   `per_device_train_batch_size`: Batch size per GPU/CPU for training.
*   `gradient_accumulation_steps`: Number of updates steps to accumulate gradients before performing a backward/update pass. Useful for simulating larger batch sizes with limited memory.
*   `optim`: Optimizer to use (e.g., "paged_adamw_8bit" for memory efficiency).
*   `logging_steps`: How often to log training metrics.
*   `learning_rate`: Initial learning rate for the optimizer.
*   `fp16`/`bf16`: Whether to use mixed precision training (float16 or bfloat16) for faster training and less memory. `bf16` is generally preferred if supported by the hardware.
*   `max_grad_norm`: Maximum gradient norm for gradient clipping.
*   `lr_scheduler_type`: Type of learning rate scheduler (e.g., "cosine").
*   `warmup_ratio`: Proportion of training steps to perform linear warmup.
*   `disable_tqdm`: Whether to disable the tqdm progress bar.
*   `evaluation_strategy`/`save_strategy`/`save_steps`: Strategies for evaluation and checkpoint saving.

In [7]:
!pip install --upgrade torchao # Added to resolve ImportError from incompatible version

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments

# 1. Prepare the model for k-bit training (Quantization specific)
# This is crucial when using BitsAndBytes for 4-bit or 8-bit quantization,
# as it casts various model components to float32 for stability.
model.gradient_checkpointing_enable() # Enable gradient checkpointing to save memory
model = prepare_model_for_kbit_training(model)

# 2. Configure LoRA
lora_config = LoraConfig(
    r=8,  # Rank of the update matrices. Lower rank means fewer trainable parameters.
    lora_alpha=16, # LoRA scaling factor. A higher alpha gives more weight to the LoRA updates.
    target_modules=["q_proj", "v_proj"], # Target specific layers for LoRA. Common for attention layers.
    lora_dropout=0.05, # Dropout probability for LoRA layers.
    bias="none", # Do not train bias parameters.
    task_type="CAUSAL_LM", # Specify the task type for the PEFT model.
)

# 3. Apply LoRA to the model
model = get_peft_model(model, lora_config)
print("LoRA model prepared successfully.")
model.print_trainable_parameters() # Print the number of trainable parameters

# Explicitly move the model to GPU after all PEFT modifications
model.to('cuda')

# 4. Configure Training Arguments
training_args = TrainingArguments(
    output_dir="./tinyllama-medical-qa", # Output directory for checkpoints and logs
    num_train_epochs=3, # Number of training epochs
    per_device_train_batch_size=2, # Batch size per device (adjust based on GPU memory)
    gradient_accumulation_steps=4, # Accumulate gradients over 4 steps to simulate a larger batch size
    optim="paged_adamw_8bit", # Optimizer, 'paged_adamw_8bit' is good for memory efficiency
    logging_steps=10, # Log training metrics every 10 steps
    learning_rate=2e-4, # Learning rate
    fp16=True, # Enable fp16 mixed precision training (changed from False)
    bf16=False, # Disable bf16 mixed precision training (changed from True)
    max_grad_norm=0.3, # Maximum gradient norm for gradient clipping
    lr_scheduler_type="cosine", # Learning rate scheduler type
    warmup_ratio=0.03, # Linear warmup ratio
    disable_tqdm=False, # Enable tqdm progress bar
    eval_strategy="no", # No evaluation during training for simplicity
    save_strategy="epoch", # Save checkpoint at the end of each epoch
    load_best_model_at_end=False, # Don't load best model at the end
    report_to="none" # Do not report to any experiment tracking platform
)

print("Training arguments configured successfully.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


LoRA model prepared successfully.
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
Training arguments configured successfully.


## Task 3: Analyze Model Parameters

**Objective**

Understand the parameter count of the base model and the LoRA-adapted model to appreciate the efficiency of PEFT.

**Deliverables**

*   Display the total parameters of the base LLM.
*   Display the trainable parameters after applying LoRA.
*   Explain why full fine-tuning is expensive.

In [8]:
# Total parameters of the original model
total_params = model.num_parameters()
print(f"Total parameters of the base model: {total_params / 1e9:.2f} billion")

Total parameters of the base model: 1.10 billion


In [9]:
# Trainable parameters after applying LoRA
# This was already printed during the LoRA setup, but re-emphasizing here.
print("Trainable parameters after LoRA (re-printed for clarity):")
model.print_trainable_parameters()

Trainable parameters after LoRA (re-printed for clarity):
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### Why Full Fine-Tuning is Expensive

Full fine-tuning involves updating all the parameters of a pretrained model. For large language models, this is incredibly resource-intensive for several reasons:

1.  **Memory Consumption:** Each parameter requires memory for its value, its gradient, and optimizer states (e.g., momentum for AdamW). For models with billions of parameters, this quickly exceeds the memory capacity of even high-end GPUs.
    *   For example, a model with 1 billion parameters, if stored in `float16` (2 bytes per parameter), would require 2GB for just the weights. If using an AdamW optimizer, each parameter might need ~12 bytes (2 for weight, 2 for gradient, 8 for optimizer states), pushing the requirement to ~12GB per billion parameters. Add activations, and it becomes impractical.

2.  **Computational Cost:** Updating billions of parameters requires massive amounts of computation during the backward pass (to calculate gradients) and the optimization step (to update weights). This translates to:
    *   **Longer Training Times:** Training can take days, weeks, or even months.
    *   **Higher Energy Consumption:** Significant energy is expended, leading to environmental impact and operational costs.

3.  **Data Requirements:** While fine-tuning generally uses less data than pretraining, full fine-tuning still benefits from larger task-specific datasets to prevent overfitting and achieve optimal performance across all updated parameters.

4.  **Risk of Catastrophic Forgetting:** When all parameters are updated, there's a higher risk of the model "forgetting" general knowledge learned during pretraining and becoming overly specialized to the fine-tuning data. LoRA helps mitigate this by keeping the original weights frozen.

## Task 6: Create Medical QA Dataset

**Objective**

Prepare a small medical question-answering dataset based on new examples.

In [10]:
from datasets import Dataset

# Create a small synthetic medical question-answering dataset based on the new examples
new_data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

dataset = Dataset.from_dict(new_data)

print(f"Created a new synthetic dataset with {len(dataset)} examples.")
print("Example entry:")
print(dataset[0])

Created a new synthetic dataset with 3 examples.
Example entry:
{'question': 'What is diabetes?', 'answer': 'Diabetes is a chronic disease where the body cannot regulate blood sugar properly.'}


Now, we need to tokenize this new dataset using the same formatting function as before.

In [11]:
def formatting_function(example):
    # Format the input and target for the LLM
    # Using a chat format similar to how TinyLlama was fine-tuned
    text = f"<|im_start|>user\n{example['question']}<|im_end|>\n<|im_start|>assistant\n{example['answer']}<|im_end|>"

    # Tokenize the formatted text, ensuring uniform length for all examples
    # by applying both truncation and padding to max_length
    tokenized_output = tokenizer(text, truncation=True, max_length=512, padding="max_length")

    # Add 'labels' which are the same as 'input_ids' for causal language modeling
    # The loss function will mask out the input tokens during training
    tokenized_output["labels"] = tokenized_output["input_ids"]
    return tokenized_output

# Apply the formatting and tokenization function to the new dataset
tokenized_dataset = dataset.map(formatting_function, batched=False)

print("Tokenized dataset example with new data:")
print(tokenized_dataset[0])
print(f"Decoded example: {tokenizer.decode(tokenized_dataset[0]['input_ids'])}")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenized dataset example with new data:
{'question': 'What is diabetes?', 'answer': 'Diabetes is a chronic disease where the body cannot regulate blood sugar properly.', 'input_ids': [1, 529, 29989, 326, 29918, 2962, 29989, 29958, 1792, 13, 5618, 338, 652, 370, 10778, 29973, 29966, 29989, 326, 29918, 355, 29989, 29958, 13, 29966, 29989, 326, 29918, 2962, 29989, 29958, 465, 22137, 13, 12130, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 19423, 29989, 326, 29918, 355, 29989, 29958, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 

### Deliverables for Task 7: Tokenization

Here are the `input_ids` and `attention_mask` for the first example in the tokenized dataset:

In [13]:
print(f"Input IDs (first example): {tokenized_dataset[0]['input_ids']}")
print(f"Attention Mask (first example): {tokenized_dataset[0]['attention_mask']}")

Input IDs (first example): [1, 529, 29989, 326, 29918, 2962, 29989, 29958, 1792, 13, 5618, 338, 652, 370, 10778, 29973, 29966, 29989, 326, 29918, 355, 29989, 29958, 13, 29966, 29989, 326, 29918, 2962, 29989, 29958, 465, 22137, 13, 12130, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 19423, 29989, 326, 29918, 355, 29989, 29958, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2

## Task 7: Fine-tune the Model

**Objective**

Fine-tune the TinyLlama model with the prepared dataset using LoRA.

In [12]:
from transformers import DataCollatorForLanguageModeling, Trainer

# Data collator for causal language modeling
# It will dynamically pad the sequences to the longest length in the batch
# and prepare the labels for causal LM (shift inputs right to align with labels)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Trainer initialized successfully. Starting training...")

# Start training
trainer.train()

print("Training complete!")

Trainer initialized successfully. Starting training...


Step,Training Loss


Training complete!


### Deliverables for Task 8: Fine-Tune Using LoRA

**Why base model weights remain frozen during LoRA fine-tuning:**

In LoRA (Low-Rank Adaptation), the vast majority of the pretrained base model's weights are kept **frozen** during the fine-tuning process. This is a core design principle that offers significant advantages:

1.  **Efficiency:** Instead of updating billions of parameters, LoRA introduces small, low-rank matrices (adapters) into specific layers of the model. Only these adapter weights are trained. This drastically reduces the number of trainable parameters, leading to:
    *   **Faster training:** Fewer computations are needed for gradient calculation and weight updates.
    *   **Lower memory consumption:** Less GPU memory is required to store gradients and optimizer states for the adapter weights, making it possible to fine-tune very large models on more modest hardware.

2.  **Preservation of Pretrained Knowledge:** By freezing the original weights, the model retains the extensive general knowledge, linguistic patterns, and factual understanding it acquired during its initial pre-training on massive datasets. This prevents "catastrophic forgetting," where fine-tuning a model on a smaller, specific dataset might erase or degrade its broader capabilities.

3.  **Modularity and Flexibility:** The LoRA adapters are separate from the base model. This means:
    *   You can create multiple adapters for different downstream tasks using the same base model.
    *   Adapters can be easily swapped in and out without needing to reload the entire base model.
    *   The base model can be updated independently, and adapters can be re-applied or retrained with minimal effort.

In essence, LoRA allows us to efficiently *adapt* the model to new tasks by adding a small, trainable "delta" to its capabilities, rather than completely re-shaping its fundamental knowledge.

### Deliverables for Task 9: Save LoRA Adapter

Let's save the LoRA adapter weights.

In [14]:
# Save the LoRA adapter weights
output_dir = "./tinyllama-medical-qa-lora-adapter"
trainer.model.save_pretrained(output_dir)

print(f"LoRA adapter weights saved to {output_dir}")

LoRA adapter weights saved to ./tinyllama-medical-qa-lora-adapter


**Why LoRA adapters are much smaller than full models:**

LoRA adapters are significantly smaller than the full pretrained models they modify for several key reasons:

1.  **Low-Rank Approximation:** The core idea behind LoRA is to approximate the weight updates during fine-tuning with low-rank matrices. Instead of learning a full, high-dimensional weight matrix \( \Delta W \), LoRA decomposes it into two smaller matrices, \( A \) and \( B \), such that \( \Delta W = BA \). If the original weight matrix has dimensions \( d_{in} \times d_{out} \), and the rank \( r \) is much smaller than both \( d_{in} \) and \( d_{out} \), then the number of parameters in \( A \) (\( d_{in} \times r \)) and \( B \) (\( r \times d_{out} \)) is substantially less than in \( \Delta W \) (\( d_{in} \times d_{out} \)).

2.  **Targeted Application:** LoRA doesn't add parameters to every layer or every part of the model. It typically targets only the attention mechanism's query and value projection matrices (e.g., `q_proj` and `v_proj`) because these are found to be highly effective for adaptation. This selective application further minimizes the additional parameters.

3.  **Freezing Base Weights:** The original, vast number of parameters in the base model (e.g., 1.1 billion for TinyLlama) are kept frozen and are not updated or saved as part of the adapter. The adapter only contains the newly introduced, small low-rank matrices.

These factors combine to make LoRA adapters extremely compact. For instance, in our case, the TinyLlama base model has 1.1 billion parameters, but the LoRA adapter introduces only 1.1 million trainable parameters, representing about 0.1% of the total model size. This tiny footprint makes LoRA adapters easy to store, share, and switch between for different tasks.

## Task 10: Medical Question Answering

**Objective**
Generate answers using the fine-tuned model.

**Deliverables**

*   Show generated response.
*   Compare: Before LoRA (Generic Response) vs. After LoRA (Medical Domain Response)

### Generate Response Before LoRA (Generic Model)

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Reload the tokenizer (if not already available)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad_token is set for the tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the *base* model without LoRA for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
base_model.eval() # Set to evaluation mode

def generate_response(question, model, tokenizer):
    # Format the input similar to training
    prompt = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,  # Limit response length
            num_return_sequences=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=True, # Enable sampling for more varied responses
            temperature=0.7, # Controls randomness, lower is less random
            top_p=0.9 # Nucleus sampling
        )

    # Decode and return the generated text, excluding the prompt part
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Find the start of the assistant's response to exclude the input prompt
    assistant_start_tag = "<|im_start|>assistant\n"
    if assistant_start_tag in generated_text:
        response = generated_text.split(assistant_start_tag)[-1].strip()
    else:
        response = generated_text.strip() # Fallback if tag is not found

    # Remove the user part if it's still there
    user_start_tag = "<|im_start|>user\n"
    if user_start_tag in response:
        response = response.split(user_start_tag)[-1].strip()

    user_end_tag = "<|im_end|>"
    if user_end_tag in response:
        response = response.split(user_end_tag)[0].strip()

    return response

# Example Question
question = "What are the symptoms of hypertension?"

# Generate response using the base model
print("Question:", question)
response_before_lora = generate_response(question, base_model, tokenizer)
print("Response (Before LoRA):", response_before_lora)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Question: What are the symptoms of hypertension?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response (Before LoRA): The symptoms of hypertension include:
1. Headache
2. Muscle pain or tension
3. Shortness of breath or difficulty breathing
4. Pain in the arms or legs
5. Swelling in the ankles or feet
6. Thirst
7. Frequent urination
8. Nausea or vomiting
9. Weakness or fatigue
10. Redness or itching in the skin


### Generate Response After LoRA (Fine-Tuned Model)

In [16]:
from peft import PeftModel, PeftConfig

# Reload the tokenizer (if not already available)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad_token is set for the tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the base model first (same as before)
base_model_for_lora = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Load the LoRA adapter weights onto the base model
lora_model = PeftModel.from_pretrained(base_model_for_lora, output_dir)
lora_model.eval() # Set to evaluation mode

# Example Question (same as before)
question = "What are the symptoms of hypertension?"

# Generate response using the LoRA fine-tuned model
print("Question:", question)
response_after_lora = generate_response(question, lora_model, tokenizer)
print("Response (After LoRA):", response_after_lora)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What are the symptoms of hypertension?
Response (After LoRA): Sure, here are some symptoms of hypertension:
- High blood pressure (hypertension)
- Shortness of breath
- Weakness or fatigue
- Rash or swelling in the face, arms, or legs
- Dizziness or fainting
- Cravings for salt or sugar
- Nausea or vomiting
- Frequent urination or excessive sleepiness
- Severe headaches or pain in the neck


### Comparison of Responses

**Question:** What are the symptoms of hypertension?

**Response (Before LoRA - Generic):**
```


In [17]:
print(response_before_lora)

The symptoms of hypertension include:
1. Headache
2. Muscle pain or tension
3. Shortness of breath or difficulty breathing
4. Pain in the arms or legs
5. Swelling in the ankles or feet
6. Thirst
7. Frequent urination
8. Nausea or vomiting
9. Weakness or fatigue
10. Redness or itching in the skin


```

**Response (After LoRA - Medical Domain):**
```


In [18]:
print(response_after_lora)

Sure, here are some symptoms of hypertension:
- High blood pressure (hypertension)
- Shortness of breath
- Weakness or fatigue
- Rash or swelling in the face, arms, or legs
- Dizziness or fainting
- Cravings for salt or sugar
- Nausea or vomiting
- Frequent urination or excessive sleepiness
- Severe headaches or pain in the neck


```

**Analysis:**

By comparing the two responses, we can observe how the fine-tuning with LoRA has adapted the model's output to be more aligned with the medical domain information present in our training data. The 'Before LoRA' response is expected to be more generic, while the 'After LoRA' response should be more specific and relevant to medical contexts, potentially directly reflecting the examples it was fine-tuned on.